# screamingface · Quickstart

Compose three models, evaluate their majority vote, and check whether the fusion beats its best
individual model.

**Path: bare quickstart · No architecture knowledge required.** This notebook stays focused on
the public compose → run → compare loop. The linked architecture notebook shows the URL4 request,
node graph, and response envelope.

This quickstart is zero-setup. By default, ScreamingFace runs a real URL4 node in-process and its
model-route leaves return deterministic local answers. URL4 still parses and executes the complete
fusion graph; no AI Gateway or provider is contacted.

To fetch GPQA Diamond instead of the bundled fixture, first accept its gated dataset terms and be
logged in to Hugging Face, then select the live dataset with `sf.config(mode="live")`. Dataset mode
and engine mode are independent, so this still uses the local mock engine unless you select an
HTTP URL4 engine explicitly.

In [1]:
import screamingface as sf

# Optional: replace the default in-process mock with an HTTP URL4 engine.
# sf.config("http://127.0.0.1:4404")  # first run ./scripts/dev-url4.sh
# sf.config("https://url4.example")

## 1 · Compose

In [2]:
fusion = sf.Fusion(
    "frontier-trio",
    models=[
        "codex/gpt-5.5",
        "gemini-cli/gemini-2.5-pro",
        "anthropic/claude-sonnet-4-6",
    ],
    reducer=sf.MajorityVote(tie_breaker="codex/gpt-5.5"),
)
fusion

Role,Model
Tie breaker,codex/gpt-5.5
Model,gemini-cli/gemini-2.5-pro
Model,anthropic/claude-sonnet-4-6


Every fusion has a shareable URL4 recipe. Displaying it does not execute it.

In [3]:
fusion.url4

"(panel_1=/codex/gpt-5.5()!'$question', panel_2=/gemini/2.5()!'$question', panel_3=/claude/sonnet-4.6()!'$question', {schema: 'screamingface.panel-result.v2', panel_1_id: 'codex/gpt-5.5', panel_1_model: 'codex/gpt-5.5', panel_1_answer: '$panel_1', panel_2_id: 'gemini-cli/gemini-2.5-pro', panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2', panel_3_id: 'anthropic/claude-sonnet-4-6', panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3'})"

## 2 · Run

In [4]:
run = fusion.evaluate("gpqa", first=20, seed=0)
run

Run(benchmark='GPQA-shaped synthetic science fixture', dataset_source='synthetic-gpqa-shaped', mode='mock', models=('codex/gpt-5.5', 'gemini-cli/gemini-2.5-pro', 'anthropic/claude-sonnet-4-6'), url="(panel_1=/codex/gpt-5.5()!'$question', panel_2=/gemini/2.5()!'$question', panel_3=/claude/sonnet-4.6()!'$question', {schema: 'screamingface.panel-result.v2', panel_1_id: 'codex/gpt-5.5', panel_1_model: 'codex/gpt-5.5', panel_1_answer: '$panel_1', panel_2_id: 'gemini-cli/gemini-2.5-pro', panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2', panel_3_id: 'anthropic/claude-sonnet-4-6', panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3'})", sample_size=20, seed=0, score=100.0, baseline=80.0, gain=20.0, cost_usd=0.0, engine='mock', fusion_name='frontier-trio', reducer='majority_vote', tie_breaker='codex/gpt-5.5', incomplete=0, profiles=(), pricing_source='engine response does not yet report usage', pricing_as_of='n/a', prompt_tokens=0, completion_tokens=0, total_tokens=0, model_results=(ModelResult(model='codex/gpt-5.5', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0, name=None, metrics=(('accuracy', 80.0),)), ModelResult(model='gemini-cli/gemini-2.5-pro', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0, name=None, metrics=(('accuracy', 80.0),)), ModelResult(model='anthropic/claude-sonnet-4-6', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0, name=None, metrics=(('accuracy', 80.0),))), failures=(), primary_metric='accuracy', metrics=(('accuracy', 100.0),))

## 3 · Compare

In [5]:
run.score, run.baseline, run.gain  # fusion, best model, improvement

(100.0, 80.0, 20.0)

> Positive gain means the fusion outperformed its strongest individual model.

For the exact URL4 HTTP request and compiled-node walkthrough, open
[`sf_url4_engine.ipynb`](sf_url4_engine.ipynb). The complete public API and execution guide is
[`../docs/index.html`](../docs/index.html).